# L3 — Set up Resources: Registry, Gateway & Cognito

## What You'll Learn

- How to create an AgentCore Registry as a centralized catalog for specialist agents
- How to enable CloudWatch Transaction Search for end-to-end agent observability
- How to create an AgentCore Gateway with Cognito JWT authentication
- How to provision test users and publish all resource IDs to SSM for downstream notebooks

## Overview

Before deploying any specialist agents, AnyCompany's multi-agent system needs two shared pieces of infrastructure: a Registry where agents can publish themselves, and a Gateway that routes incoming requests to the right agent. This notebook creates both, along with the Cognito User Pool that authenticates chatbot users.

The Registry starts empty. Notebooks 2 and 3 each deploy a specialist agent and register it here. Once registered, the Harness (Notebook 4) discovers them dynamically — no hardcoded endpoints. All resource IDs are stored in SSM Parameter Store so every subsequent notebook can look them up without re-creating anything.

## Architecture

```
  ┌─────────────────────────────────────────────────────────────┐
  │  Created in this notebook                                    │
  │                                                             │
  │  AgentCore Registry  ◄── specialist agents register here   │
  │  (anycompany-agent-registry, auto-approval enabled)         │
  │                                                             │
  │  AgentCore Gateway   ◄── all tool calls route through here │
  │  (MCP protocol, Cognito JWT auth)                           │
  │       │                                                     │
  │       └── IAM Role → invoke order + refund Lambda tools     │
  │                                                             │
  │  Cognito User Pool                                          │
  │  ├── gold_customer   (CUST-789)                             │
  │  ├── silver_customer (CUST-456)                             │
  │  └── bronze_customer (CUST-123)                             │
  └─────────────────────────────────────────────────────────────┘
                    │
                    ▼
              SSM Parameter Store
              /anycompany/agentcore/...
              (consumed by Notebooks 2–5 and L4, L5)
```

## Steps

1. Enable CloudWatch Transaction Search
2. Create the AgentCore Registry
3. Create the Cognito User Pool and test users
4. Create the AgentCore Gateway
5. Save all resource IDs to SSM Parameter Store

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- AWS credentials configured with permissions for AgentCore, Cognito, IAM, Lambda, CloudWatch, X-Ray, and SSM
- `boto3` (installed in the first code cell)
- L2 notebook completed — `model_id` and `guardrail_id` must be in SSM before running L3 agents
- This notebook must run before Notebooks 2, 3, and 4

Install required packages.

In [ ]:
%pip install --quiet boto3==1.43.0

Import libraries, set the region, and initialise AWS clients.

In [ ]:
import boto3
import json
import time
import botocore
import os

# --- Configuration ---
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1") # Change to your preferred region
REGISTRY_NAME = "anycompany-agent-registry"
GATEWAY_NAME = "anycompany-agent-gateway"

# Cognito User Pool config (for Gateway OAuth auth)
# These will be created in Notebook 6, but set them here if you already have them
COGNITO_USER_POOL_ID = os.environ.get("COGNITO_USER_POOL_ID", "")  # e.g. us-east-1_xxxxxxx
COGNITO_CLIENT_ID = os.environ.get("COGNITO_CLIENT_ID", "")  # App client ID

# Clients
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)

ACCOUNT_ID = sts.get_caller_identity()["Account"]
print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")

## Step 1: Enable CloudWatch Transaction Search

One-time account/region setup that routes X-Ray trace spans to CloudWatch Logs, enabling the GenAI Observability dashboard in L5.

Create the X-Ray resource policy, set trace destination to CloudWatch Logs, enable Application Signals, and verify status.

In [ ]:
# Enable CloudWatch Transaction Search for AgentCore Observability
logs_client = boto3.client("logs", region_name=REGION)
xray = boto3.client("xray", region_name=REGION)
app_signals = boto3.client("application-signals", region_name=REGION)

# 1. Resource policy: allow X-Ray to write spans to CloudWatch Logs
logs_client.put_resource_policy(
    policyName="TransactionSearchXRayAccess",
    policyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Sid": "TransactionSearchXRayAccess",
            "Effect": "Allow",
            "Principal": {"Service": "xray.amazonaws.com"},
            "Action": "logs:PutLogEvents",
            "Resource": [
                f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:aws/spans:*",
                f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/application-signals/data:*"
            ],
            "Condition": {
                "ArnLike": {"aws:SourceArn": f"arn:aws:xray:{REGION}:{ACCOUNT_ID}:*"},
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID}
            }
        }]
    })
)
print("✓ Resource policy created")

# 2. Route trace segments to CloudWatch Logs
try:
    xray.update_trace_segment_destination(Destination="CloudWatchLogs")
    print("✓ Trace destination set to CloudWatchLogs")
except xray.exceptions.InvalidRequestException:
    print("✓ Trace destination already set to CloudWatchLogs")

# 3. Index 1% of spans (free tier)
xray.update_indexing_rule(Name="Default", Rule={"Probabilistic": {"DesiredSamplingPercentage": 1}})
print("✓ Indexing rule set to 1%")

# 4. Enable Application Signals
app_signals.start_discovery()
print("✓ Application Signals enabled")

# 5. Verify
resp = xray.get_trace_segment_destination()
status = resp['Status']
print(f"\nTransaction Search: {resp['Destination']} ({status})")
if status == 'PENDING':
    print("\n⏳ Status is PENDING. This can take up to 10 minutes.")
    print("   You can proceed with the next steps — Transaction Search will be ready by the time you invoke agents.")
elif status == 'ACTIVE':
    print("\n✓ Transaction Search is ACTIVE")

## Step 2: Create the AgentCore Registry

Centralized catalog where specialist agents publish their metadata. Auto-approval is enabled so agents become discoverable immediately on registration.

NOTE: It takes typically 3-5 minutes to create registry

In [ ]:
print(f"Creating registry: {REGISTRY_NAME}\n")

try:
    resp = agentcore_control.create_registry(
        name=REGISTRY_NAME,
        description="AnyCompany agent registry — centralized catalog for order, refund, and vision agents.",
        approvalConfiguration={"autoApproval": True},
    )

    REGISTRY_ARN = resp["registryArn"]
    REGISTRY_ID = REGISTRY_ARN.split("/")[-1]
    print(f"Created registry: {REGISTRY_NAME} (ID: {REGISTRY_ID})")
    print(f"  ARN:           {REGISTRY_ARN}")
    print(f"  Auto-approval: Enabled")

except agentcore_control.exceptions.ConflictException:
    print(f"Registry '{REGISTRY_NAME}' already exists. Looking up by name...")
    registries = agentcore_control.list_registries()
    registry_match = None
    for reg in registries.get("registries", []):
        if reg.get("name") == REGISTRY_NAME:
            registry_match = reg
            break
    if registry_match:
        REGISTRY_ARN = registry_match["registryArn"]
        REGISTRY_ID = REGISTRY_ARN.split("/")[-1]
        print(f"  ID:     {REGISTRY_ID}")
        print(f"  ARN:    {REGISTRY_ARN}")
        print(f"  Status: {registry_match.get('status')}")
    else:
        raise RuntimeError(f"Could not find registry with name '{REGISTRY_NAME}'")

Poll until the Registry reaches READY status.

In [ ]:
# Poll until the registry transitions from CREATING to READY
import time as _time

print("Waiting for registry to be ready...")
_reg_start = _time.time()
_poll = 0

while True:
    r = agentcore_control.get_registry(registryId=REGISTRY_ID)
    _elapsed = _time.time() - _reg_start
    _poll += 1
    if r["status"] == "READY":
        print(f"Registry is READY after {_elapsed:.1f}s ({_poll} poll(s))")
        break
    print(f"  [{_elapsed:5.1f}s] Status: {r['status']} - waiting... (poll #{_poll})")
    time.sleep(3)

print(f"\nRegistry ID:  {REGISTRY_ID}")
print(f"Registry ARN: {REGISTRY_ARN}")

Confirm the Registry is empty — specialist agents will register in Notebooks 2 and 3.

In [ ]:
records_response = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
records = records_response.get("registryRecords", [])
print(f"Registry records: {len(records)}")
if len(records) == 0:
    print("Registry is empty — specialist agents will register in Notebooks 2-4.")
else:
    for r in records:
        print(f"  - {r.get('name', 'unknown')} ({r.get('status', 'unknown')})")

## Step 3: Create the Cognito User Pool

Authenticates chatbot users. The Gateway validates JWT tokens from this pool via OIDC discovery.

Create the Cognito User Pool and app client, or reuse existing ones. 

The Cognito pool configuration is for demonstration purposes only and should not be used for production.

In [ ]:
cognito = boto3.client("cognito-idp", region_name=REGION)
POOL_NAME = "AnyCompanyAgentPool"

if not COGNITO_USER_POOL_ID:
    pool_resp = cognito.create_user_pool(
        PoolName=POOL_NAME,
        Policies={
            "PasswordPolicy": {
                "MinimumLength": 12,
                "RequireUppercase": True,
                "RequireLowercase": True,
                "RequireNumbers": True,
                "RequireSymbols": True,
            }
        },
        AutoVerifiedAttributes=["email"],
        Schema=[
            {"Name": "email", "Required": True, "Mutable": True},
            {"Name": "customer_id", "AttributeDataType": "String", "Mutable": True, "StringAttributeConstraints": {"MinLength": "1", "MaxLength": "50"}},
            {"Name": "tier", "AttributeDataType": "String", "Mutable": True, "StringAttributeConstraints": {"MinLength": "1", "MaxLength": "20"}},
        ],
    )
    COGNITO_USER_POOL_ID = pool_resp["UserPool"]["Id"]
    print(f"Created User Pool: {COGNITO_USER_POOL_ID}")

    client_resp = cognito.create_user_pool_client(
        UserPoolId=COGNITO_USER_POOL_ID,
        ClientName="AnyCompanyChatbot",
        ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
        GenerateSecret=False,
    )
    COGNITO_CLIENT_ID = client_resp["UserPoolClient"]["ClientId"]
    print(f"Created App Client: {COGNITO_CLIENT_ID}")
else:
    print(f"Using existing pool: {COGNITO_USER_POOL_ID}")
    if not COGNITO_CLIENT_ID:
        clients = cognito.list_user_pool_clients(UserPoolId=COGNITO_USER_POOL_ID, MaxResults=10)
        COGNITO_CLIENT_ID = clients["UserPoolClients"][0]["ClientId"]
    print(f"Using existing client: {COGNITO_CLIENT_ID}")

print(f"\nCognito User Pool ID: {COGNITO_USER_POOL_ID}")
print(f"Cognito Client ID:    {COGNITO_CLIENT_ID}")

Create three test users (gold, silver, bronze) with workshop credentials. Password is stored in SSM as a SecureString immediately after.

In [ ]:
# Generate a strong workshop password (or reuse the one already in SSM from a previous run).
# All three test users share this password; it is stored in SSM as a SecureString below.
import secrets, string

_ssm_for_pw = boto3.client("ssm", region_name=REGION)
_SSM_PREFIX_LOCAL = "/anycompany/agentcore"
_SSM_PASSWORD_PATH = f"{_SSM_PREFIX_LOCAL}/user_password"

def _generate_password(length: int = 16) -> str:
    # Cognito requires at least one of upper, lower, digit, symbol — seed one of each, fill, shuffle.
    upper  = secrets.choice(string.ascii_uppercase)
    lower  = secrets.choice(string.ascii_lowercase)
    digit  = secrets.choice(string.digits)
    symbol = secrets.choice("!@#$%^&*")
    rest   = [secrets.choice(string.ascii_letters + string.digits + "!@#$%^&*") for _ in range(length - 4)]
    pool   = [upper, lower, digit, symbol] + rest
    secrets.SystemRandom().shuffle(pool)
    return "".join(pool)

try:
    USER_PASSWORD = _ssm_for_pw.get_parameter(Name=_SSM_PASSWORD_PATH, WithDecryption=True)["Parameter"]["Value"]
    print(f"Reusing existing workshop password from SSM: {_SSM_PASSWORD_PATH}")
except _ssm_for_pw.exceptions.ParameterNotFound:
    USER_PASSWORD = _generate_password()
    print(f"Generated new workshop password (will be stored at: {_SSM_PASSWORD_PATH})")

TEST_USERS = [
    {"username": "gold_customer", "password": USER_PASSWORD, "email": "gold@example.com", "customer_id": "CUST-789", "tier": "gold"},
    {"username": "silver_customer", "password": USER_PASSWORD, "email": "silver@example.com", "customer_id": "CUST-456", "tier": "silver"},
    {"username": "bronze_customer", "password": USER_PASSWORD, "email": "bronze@example.com", "customer_id": "CUST-123", "tier": "bronze"},
]

for user in TEST_USERS:
    try:
        cognito.admin_create_user(
            UserPoolId=COGNITO_USER_POOL_ID,
            Username=user["username"],
            UserAttributes=[
                {"Name": "email", "Value": user["email"]},
                {"Name": "email_verified", "Value": "true"},
                {"Name": "custom:customer_id", "Value": user["customer_id"]},
                {"Name": "custom:tier", "Value": user["tier"]},
            ],
            MessageAction="SUPPRESS",
        )
        cognito.admin_set_user_password(
            UserPoolId=COGNITO_USER_POOL_ID,
            Username=user["username"],
            Password=user["password"],
            Permanent=True,
        )
        print(f"  Created: {user['username']} ({user['tier']} tier, {user['customer_id']})")
    except cognito.exceptions.UsernameExistsException:
        print(f"  Exists:  {user['username']}")

print("\nTest users ready.")

## Step 4: Create the AgentCore Gateway

Entry point for all tool calls. Uses MCP protocol with Cognito JWT authentication.

Create the IAM role the Gateway will assume to invoke the order and refund Lambda tools.

In [ ]:
iam = boto3.client("iam")
GATEWAY_ROLE_NAME = "AnyCompanyAgentCoreGatewayRole"

gateway_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

gateway_permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "InvokeLambdaTargets",
            "Effect": "Allow",
            "Action": [
                "lambda:InvokeFunction"
            ],
            "Resource": [
                f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:anycompany_order_tools",
                f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:anycompany_refund_tools"
            ]
        }
    ]
}

try:
    role_response = iam.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(gateway_trust_policy),
        Description="Execution role for AnyCompany AgentCore Gateway"
    )
    GATEWAY_ROLE_ARN = role_response["Role"]["Arn"]
    print(f"Created role: {GATEWAY_ROLE_ARN}")

    iam.put_role_policy(
        RoleName=GATEWAY_ROLE_NAME,
        PolicyName="AnyCompanyGatewayPermissions",
        PolicyDocument=json.dumps(gateway_permissions_policy)
    )
    print("Attached inline policy.")

    # IAM role propagation takes a few seconds
    print("Waiting 10s for IAM propagation...")
    time.sleep(10)

except iam.exceptions.EntityAlreadyExistsException:
    GATEWAY_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{GATEWAY_ROLE_NAME}"
    print(f"Role already exists: {GATEWAY_ROLE_ARN}")

print(f"\nGateway Role ARN: {GATEWAY_ROLE_ARN}")

Create the Gateway with MCP protocol and Cognito JWT auth, or look it up if it already exists.

In [ ]:
print(f"Creating gateway: {GATEWAY_NAME}\n")

try:
    resp = agentcore_control.create_gateway(
        name=GATEWAY_NAME,
        description="AnyCompany agent gateway — routes requests to specialist agents via the registry.",
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        **({
            "authorizerType": "CUSTOM_JWT",
            "authorizerConfiguration": {
                "customJWTAuthorizer": {
                    "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_USER_POOL_ID}/.well-known/openid-configuration",
                    "allowedAudience": [COGNITO_CLIENT_ID],
                }
            },
        } if COGNITO_USER_POOL_ID else {"authorizerType": "AWS_IAM"}),
    )

    GATEWAY_ID = resp["gatewayId"]
    GATEWAY_URL = resp["gatewayUrl"]
    GATEWAY_ARN = resp.get("gatewayArn", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}")
    print(f"Created gateway: {GATEWAY_NAME}")
    print(f"  ID:  {GATEWAY_ID}")
    print(f"  URL: {GATEWAY_URL}")
    print(f"  ARN: {GATEWAY_ARN}")

except agentcore_control.exceptions.ConflictException:
    print(f"Gateway '{GATEWAY_NAME}' already exists. Looking up by name...")
    gateways = agentcore_control.list_gateways()
    gateway_match = None
    for gw in gateways.get("gateways", []):
        if gw.get("name") == GATEWAY_NAME:
            gateway_match = gw
            break
    if gateway_match:
        GATEWAY_ID = gateway_match["gatewayId"]
        GATEWAY_URL = gateway_match.get("gatewayUrl", "")
        GATEWAY_ARN = gateway_match.get("gatewayArn", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}")
        print(f"  ID:     {GATEWAY_ID}")
        print(f"  URL:    {GATEWAY_URL}")
        print(f"  ARN:    {GATEWAY_ARN}")
        print(f"  Status: {gateway_match.get('status')}")
    else:
        raise RuntimeError(f"Could not find gateway with name '{GATEWAY_NAME}'")

Poll until the Gateway reaches READY status.

In [ ]:
# Poll until the gateway transitions to READY
import time
import time as _time

print("Waiting for gateway to be ready...")
_gw_start = _time.time()
_poll = 0

while True:
    g = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID)
    status = g.get("status")
    _elapsed = _time.time() - _gw_start
    _poll += 1

    if status == "READY":
        print(f"Gateway is READY after {_elapsed:.1f}s ({_poll} poll(s))")
        break

    if status in {"FAILED", "UPDATE_UNSUCCESSFUL", "DELETING"}:
        reasons = g.get("statusReasons", [])
        raise RuntimeError(f"Gateway entered terminal state {status}: {reasons}")

    print(f"  [{_elapsed:5.1f}s] Status: {status} - waiting... (poll #{_poll})")
    time.sleep(3)

print(f"\nGateway ID:  {GATEWAY_ID}")
print(f"Gateway URL: {GATEWAY_URL}")
print(f"Gateway ARN: {GATEWAY_ARN}")

## Step 5: Save Configuration to SSM Parameter Store

Publish all resource IDs to SSM so Notebooks 2–5 and L4/L5 can read them without re-creating resources.

In [ ]:
ssm = boto3.client("ssm", region_name=REGION)

SSM_PREFIX = "/anycompany/agentcore"

parameters = {
    f"{SSM_PREFIX}/region": REGION,
    f"{SSM_PREFIX}/account_id": ACCOUNT_ID,
    f"{SSM_PREFIX}/registry_id": REGISTRY_ID,
    f"{SSM_PREFIX}/registry_arn": REGISTRY_ARN,
    f"{SSM_PREFIX}/registry_name": REGISTRY_NAME,
    f"{SSM_PREFIX}/gateway_id": GATEWAY_ID,
    f"{SSM_PREFIX}/gateway_arn": GATEWAY_ARN,
    f"{SSM_PREFIX}/gateway_url": GATEWAY_URL,
    f"{SSM_PREFIX}/gateway_name": GATEWAY_NAME,
    f"{SSM_PREFIX}/gateway_role_arn": GATEWAY_ROLE_ARN,
    f"{SSM_PREFIX}/cognito_user_pool_id": COGNITO_USER_POOL_ID,
    f"{SSM_PREFIX}/cognito_client_id": COGNITO_CLIENT_ID,
}

# Sensitive values stored as SecureString (encrypted at rest with KMS).
# Consumers must call get_parameter(..., WithDecryption=True) to read them.
secure_parameters = {
    f"{SSM_PREFIX}/user_password": USER_PASSWORD,
}

print(f"Saving {len(parameters)} String + {len(secure_parameters)} SecureString parameters to SSM under '{SSM_PREFIX}/'\n")

for name, value in parameters.items():
    ssm.put_parameter(
        Name=name,
        Value=value,
        Type="String",
        Overwrite=True,
    )
    print(f"  ✓ {name} = {value}")

for name, value in secure_parameters.items():
    ssm.put_parameter(
        Name=name,
        Value=value,
        Type="SecureString",
        Overwrite=True,
    )
    print(f"  ✓ {name} = *** (SecureString)")

print(f"\nAll parameters saved to SSM under prefix: {SSM_PREFIX}/")

## Summary

| Resource | Name | Purpose |
|----------|------|---------|
| AgentCore Registry | `anycompany-agent-registry` | Specialist agents publish themselves here at deploy time |
| AgentCore Gateway | `anycompany-agent-gateway` | Routes all tool calls to the correct Lambda target |
| Cognito User Pool | `AnyCompanyAgentPool` | Authenticates chatbot users via JWT |
| SSM parameters | `/anycompany/agentcore/...` | Shared config consumed by Notebooks 2–5, L4, and L5 |

### What's Next

| Notebook | What It Does |
|----------|--------------|
| `2_order_agent.ipynb` | Deploys the Order Agent and registers it in the Registry |
| `3_refund_agent.ipynb` | Deploys the Refund Agent and registers it in the Registry |
| `4_orchestrator_agent.ipynb` | Creates the Harness that discovers and delegates to both agents |

---
## Cleanup

Run the following cell to delete all resources created in this notebook and avoid ongoing charges.

**Resources deleted:**
- AgentCore Registry and Gateway
- Cognito User Pool
- Gateway IAM role
- SSM parameters

Uncomment and run to delete all resources created in this notebook.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Account-level changes from Step 1 (X-Ray Transaction Search resource policy,
##       trace destination, indexing rule, Application Signals discovery) are
##       intentionally NOT reverted — they are idempotent and useful to keep enabled.

# import boto3, os, time
# from botocore.exceptions import ClientError
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
# REGISTRY_NAME = "anycompany-agent-registry"
# GATEWAY_NAME = "anycompany-agent-gateway"
# POOL_NAME = "AnyCompanyAgentPool"
# GATEWAY_ROLE_NAME = "AnyCompanyAgentCoreGatewayRole"
#
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# cognito = boto3.client("cognito-idp", region_name=REGION)
# iam = boto3.client("iam")
# ssm = boto3.client("ssm", region_name=REGION)
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# # Resolve IDs from SSM, falling back to lookup-by-name (idempotent on partial re-runs).
# GATEWAY_ID = _ssm_get("gateway_id")
# if not GATEWAY_ID:
#     for gw in agentcore_control.list_gateways().get("gateways", []):
#         if gw.get("name") == GATEWAY_NAME:
#             GATEWAY_ID = gw["gatewayId"]
#             break
#
# REGISTRY_ID = _ssm_get("registry_id")
# if not REGISTRY_ID:
#     for reg in agentcore_control.list_registries().get("registries", []):
#         if reg.get("name") == REGISTRY_NAME:
#             REGISTRY_ID = reg["registryArn"].split("/")[-1]
#             break
#
# COGNITO_USER_POOL_ID = _ssm_get("cognito_user_pool_id")
# if not COGNITO_USER_POOL_ID:
#     pools = cognito.list_user_pools(MaxResults=60).get("UserPools", [])
#     for p in pools:
#         if p.get("Name") == POOL_NAME:
#             COGNITO_USER_POOL_ID = p["Id"]
#             break
#
# # ── Phase 1: Drain registry records (downstream notebooks register agents here) ──
# if REGISTRY_ID:
#     try:
#         records = agentcore_control.list_registry_records(registryId=REGISTRY_ID).get("registryRecords", [])
#         for rec in records:
#             try:
#                 agentcore_control.delete_registry_record(
#                     registryId=REGISTRY_ID,
#                     recordId=rec["recordId"],
#                 )
#                 print(f"Deleted registry record: {rec.get('name', rec['recordId'])}")
#             except Exception as e:
#                 print(f"Registry record cleanup ({rec.get('name')}): {e}")
#         if records:
#             time.sleep(5)  # let async record deletes propagate before deleting registry
#     except Exception as e:
#         print(f"List registry records: {e}")
#
# # ── Phase 2: Delete Gateway and poll until gone ─────────────────────────────────
# if GATEWAY_ID:
#     try:
#         agentcore_control.delete_gateway(gatewayIdentifier=GATEWAY_ID)
#         print(f"Initiated delete of Gateway: {GATEWAY_ID}")
#         deadline = time.time() + 300  # 5 min
#         while time.time() < deadline:
#             try:
#                 status = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID).get("status")
#                 print(f"  Gateway status: {status}")
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print(f"Gateway {GATEWAY_ID} fully deleted.")
#                 break
#             time.sleep(10)
#         else:
#             print("Timed out waiting for Gateway to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Gateway {GATEWAY_ID} already deleted.")
#     except Exception as e:
#         print(f"Gateway cleanup: {e}")
#
# # ── Phase 3: Delete Registry (records were drained in Phase 1) ──────────────────
# if REGISTRY_ID:
#     try:
#         agentcore_control.delete_registry(registryId=REGISTRY_ID)
#         print(f"Initiated delete of Registry: {REGISTRY_ID}")
#         deadline = time.time() + 300
#         while time.time() < deadline:
#             try:
#                 status = agentcore_control.get_registry(registryId=REGISTRY_ID).get("status")
#                 print(f"  Registry status: {status}")
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print(f"Registry {REGISTRY_ID} fully deleted.")
#                 break
#             time.sleep(10)
#         else:
#             print("Timed out waiting for Registry to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Registry {REGISTRY_ID} already deleted.")
#     except Exception as e:
#         print(f"Registry cleanup: {e}")
#
# # ── Phase 4: Delete Cognito User Pool (cascades to users and app client) ────────
# if COGNITO_USER_POOL_ID:
#     try:
#         cognito.delete_user_pool(UserPoolId=COGNITO_USER_POOL_ID)
#         print(f"Deleted Cognito User Pool: {COGNITO_USER_POOL_ID}")
#     except cognito.exceptions.ResourceNotFoundException:
#         print(f"Cognito User Pool {COGNITO_USER_POOL_ID} already deleted.")
#     except Exception as e:
#         print(f"Cognito cleanup: {e}")
#
# # ── Phase 5: Delete Gateway IAM role (after Gateway is gone — Phase 2) ──────────
# try:
#     for p in iam.list_role_policies(RoleName=GATEWAY_ROLE_NAME).get("PolicyNames", []):
#         iam.delete_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName=p)
#     for ap in iam.list_attached_role_policies(RoleName=GATEWAY_ROLE_NAME).get("AttachedPolicies", []):
#         iam.detach_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyArn=ap["PolicyArn"])
#     iam.delete_role(RoleName=GATEWAY_ROLE_NAME)
#     print(f"Deleted IAM role: {GATEWAY_ROLE_NAME}")
# except iam.exceptions.NoSuchEntityException:
#     print(f"IAM role {GATEWAY_ROLE_NAME} already deleted.")
# except Exception as e:
#     print(f"IAM cleanup: {e}")
#
# # ── Phase 6: Delete SSM parameters (last) ───────────────────────────────────────
# ssm_params = [
#     "region", "account_id",
#     "registry_id", "registry_arn", "registry_name",
#     "gateway_id", "gateway_arn", "gateway_url", "gateway_name", "gateway_role_arn",
#     "cognito_user_pool_id", "cognito_client_id",
#     "user_password",
# ]
# for param in ssm_params:
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{param}")
#         print(f"  Deleted: {SSM_PREFIX}/{param}")
#     except ssm.exceptions.ParameterNotFound:
#         pass  # already gone — re-run safe
#     except Exception as e:
#         print(f"  Failed to delete {SSM_PREFIX}/{param}: {e}")
#
# print("\n✅ All pre-requisite resources cleaned up.")
